In [ ]:
import pandas as pd
import requests

# The official backend vector layer url discovered via developer tools
URL = "https://fire.boku.ac.at/firedb/webgis/GetWaldbrandData"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json, text/javascript, */*; q=0.01",
    "X-Requested-With": "XMLHttpRequest",
}

print("Initiating raw data download from BOKU Geo-GIS servers...")

try:
    # Query the live background layer payload directly
    response = requests.get(URL, headers=headers, timeout=55)

    if response.status_code == 200:
        geojson_data = response.json()
        
        # Isolate the core array list containing the attributes
        features = geojson_data.get("features", [])

        if not features:
            print(
                "Connected successfully, but no incidents were found in the server packet."
            )
            exit()

        parsed_records = []

        print(f"Extracting properties for {len(features)} recorded fire anomalies...")

        for feature in features:
            # Isolate attributes (district, ignition type, fire crew counts)
            properties = feature.get("properties", {})

            # Isolate geographical GIS spatial coordinates
            geometry = feature.get("geometry", {})
            coordinates = geometry.get("coordinates", [None, None])

            # Unpack spatial properties
            properties["longitude"] = coordinates[0]
            properties["latitude"] = coordinates[1]

            parsed_records.append(properties)

        # Build structural dataframe matrix
        df = pd.DataFrame(parsed_records)

        # Optional: Clean placeholder flags (-9999 means missing or unrecorded value)
        # df = df.replace(-9999, pd.NA)

        # Output to CSV file layout
        output_csv = "boku_waldbrand_complete_01-11-2026.csv"
        df.to_csv(output_csv, index=False, encoding="utf-8-sig")

        print("\n" + "=" * 50)
        print(f"SUCCESS! Extracted {len(df)} forest fire records.")
        print(f"Database layer saved safely to: {output_csv}")
        print("=" * 50)

        # Output a short summary matrix tracking key metadata parameters
        print("\nSample Preview of Collected Columns:")
        print(df[["jahr", "bundesland", "gemeinde_neu", "brandflaeche"]].head())

    else:
        print(f"Server communication drop. HTTP Status: {response.status_code}")

except Exception as e:
    print(f"System error during parsing compilation: {e}")


Initiating raw data download from BOKU Geo-GIS servers...
Extracting properties for 5000 recorded fire anomalies...

SUCCESS! Extracted 5000 forest fire records.
Database layer saved safely to: boku_waldbrand_complete_01-11-2026.csv

Sample Preview of Collected Columns:
   jahr bundesland gemeinde_neu  brandflaeche
0  2022          K     Mühldorf         25940
1  2002          K      Lendorf         -9999
2  2025       STMK  Edelschrott            10
3  2017       STMK       Leoben           500
4  2020          W         Wien         -9999
